In [2]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# Cell 2: Load and Explore Data
# Load the provided datasets
daily_steps = pd.read_csv('dailySteps_merged.csv')
sleep_data = pd.read_csv('sleepDay_merged.csv')
weight_data = pd.read_csv('weightLogInfo_merged.csv')

print("Dataset Shapes:")
print(f"Daily Steps: {daily_steps.shape}")
print(f"Sleep Data: {sleep_data.shape}")
print(f"Weight Data: {weight_data.shape}")

print("\nFirst few rows of each dataset:")
print("\nDaily Steps:")
print(daily_steps.head())
print("\nSleep Data:")
print(sleep_data.head())
print("\nWeight Data:")
print(weight_data.head())

FileNotFoundError: [Errno 2] No such file or directory: 'dailySteps_merged.csv'

In [ ]:
# Cell 3: Data Cleaning and Preprocessing
# Convert date columns to datetime
daily_steps['ActivityDay'] = pd.to_datetime(daily_steps['ActivityDay'])
sleep_data['SleepDay'] = pd.to_datetime(sleep_data['SleepDay'].str.replace(' 12:00:00 AM', ''))
weight_data['Date'] = pd.to_datetime(weight_data['Date'])

# Extract day of week and month
daily_steps['DayOfWeek'] = daily_steps['ActivityDay'].dt.day_name()
daily_steps['Month'] = daily_steps['ActivityDay'].dt.month_name()

sleep_data['DayOfWeek'] = sleep_data['SleepDay'].dt.day_name()
sleep_data['Month'] = sleep_data['SleepDay'].dt.month_name()

print("Date conversion completed!")
print(f"Daily Steps date range: {daily_steps['ActivityDay'].min()} to {daily_steps['ActivityDay'].max()}")
print(f"Sleep Data date range: {sleep_data['SleepDay'].min()} to {sleep_data['SleepDay'].max()}")

In [ ]:
# Cell 4: Data Quality Check
def data_quality_report(df, df_name):
    print(f"\n=== Data Quality Report for {df_name} ===")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Missing Values:")
    print(df.isnull().sum())
    print(f"Duplicate Rows: {df.duplicated().sum()}")
    print(f"Unique Users: {df['Id'].nunique()}")
    
data_quality_report(daily_steps, "Daily Steps")
data_quality_report(sleep_data, "Sleep Data")
data_quality_report(weight_data, "Weight Data")

In [ ]:
# Cell 5: Basic Statistics
print("=== Daily Steps Statistics ===")
print(daily_steps['StepTotal'].describe())

print("\n=== Sleep Data Statistics ===")
print(sleep_data[['TotalMinutesAsleep', 'TotalTimeInBed']].describe())

print("\n=== Weight Data Statistics ===")
print(weight_data[['WeightKg', 'BMI']].describe())

In [ ]:
# Cell 6: Weekly Activity Analysis
# Average steps by day of week
weekly_steps = daily_steps.groupby('DayOfWeek')['StepTotal'].agg(['mean', 'median', 'std']).reindex([
    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
])

print("Average Steps by Day of Week:")
print(weekly_steps)

# Visualization
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
weekly_steps['mean'].plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Average Steps by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average Steps')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
# Steps distribution
plt.hist(daily_steps['StepTotal'], bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
plt.axvline(daily_steps['StepTotal'].mean(), color='red', linestyle='--', label=f'Mean: {daily_steps["StepTotal"].mean():.0f}')
plt.axvline(10000, color='blue', linestyle='--', label='Recommended: 10,000')
plt.title('Distribution of Daily Steps')
plt.xlabel('Steps')
plt.ylabel('Frequency')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: User Activity Patterns
# Top active users
user_activity = daily_steps.groupby('Id').agg({
    'StepTotal': ['mean', 'max', 'min', 'count'],
    'ActivityDay': ['min', 'max']
}).round(0)

user_activity.columns = ['Avg_Steps', 'Max_Steps', 'Min_Steps', 'Days_Recorded', 'First_Date', 'Last_Date']
user_activity = user_activity.sort_values('Avg_Steps', ascending=False)

print("Top 10 Most Active Users:")
print(user_activity.head(10))

# Visualization
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
# User activity levels
activity_levels = pd.cut(user_activity['Avg_Steps'], 
                       bins=[0, 5000, 7500, 10000, float('inf')],
                       labels=['Sedentary', 'Lightly Active', 'Active', 'Very Active'])
activity_levels.value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['lightcoral', 'lightyellow', 'lightgreen', 'lightblue'])
plt.title('User Activity Levels Distribution')

plt.subplot(1, 3, 2)
# Days recorded per user
user_activity['Days_Recorded'].hist(bins=20, color='lightseagreen', edgecolor='black')
plt.title('Days Recorded per User')
plt.xlabel('Days Recorded')
plt.ylabel('Number of Users')

plt.subplot(1, 3, 3)
# Average steps distribution
user_activity['Avg_Steps'].hist(bins=20, color='coral', edgecolor='black')
plt.axvline(user_activity['Avg_Steps'].mean(), color='red', linestyle='--', label=f'Mean: {user_activity["Avg_Steps"].mean():.0f}')
plt.axvline(10000, color='blue', linestyle='--', label='Recommended: 10,000')
plt.title('Average Steps per User Distribution')
plt.xlabel('Average Steps')
plt.ylabel('Number of Users')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Sleep Analysis
# Sleep patterns analysis
sleep_summary = sleep_data.groupby('DayOfWeek').agg({
    'TotalMinutesAsleep': 'mean',
    'TotalTimeInBed': 'mean'
}).reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

# Convert minutes to hours
sleep_summary['Avg_Sleep_Hours'] = sleep_summary['TotalMinutesAsleep'] / 60
sleep_summary['Avg_TimeInBed_Hours'] = sleep_summary['TotalTimeInBed'] / 60

print("Sleep Patterns by Day of Week:")
print(sleep_summary)

# Visualization
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sleep_summary['Avg_Sleep_Hours'].plot(kind='bar', color='purple', alpha=0.7)
plt.axhline(7, color='red', linestyle='--', label='Recommended: 7 hours')
plt.title('Average Sleep Hours by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Hours')
plt.xticks(rotation=45)
plt.legend()

plt.subplot(1, 3, 2)
# Sleep efficiency
sleep_data['SleepEfficiency'] = (sleep_data['TotalMinutesAsleep'] / sleep_data['TotalTimeInBed']) * 100
plt.hist(sleep_data['SleepEfficiency'], bins=20, color='violet', alpha=0.7, edgecolor='black')
plt.axvline(sleep_data['SleepEfficiency'].mean(), color='red', linestyle='--', 
           label=f'Mean: {sleep_data["SleepEfficiency"].mean():.1f}%')
plt.title('Sleep Efficiency Distribution')
plt.xlabel('Sleep Efficiency (%)')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(1, 3, 3)
# Sleep vs Time in Bed
plt.scatter(sleep_data['TotalTimeInBed'], sleep_data['TotalMinutesAsleep'], alpha=0.6, color='blue')
plt.plot([0, 1000], [0, 1000], 'r--', alpha=0.8, label='Perfect Efficiency')
plt.xlabel('Time in Bed (minutes)')
plt.ylabel('Time Asleep (minutes)')
plt.title('Time in Bed vs Time Asleep')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Weight and BMI Analysis
if not weight_data.empty:
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    # Weight distribution
    plt.hist(weight_data['WeightKg'], bins=15, color='orange', alpha=0.7, edgecolor='black')
    plt.axvline(weight_data['WeightKg'].mean(), color='red', linestyle='--', 
                label=f'Mean: {weight_data["WeightKg"].mean():.1f} kg')
    plt.title('Weight Distribution')
    plt.xlabel('Weight (kg)')
    plt.ylabel('Frequency')
    plt.legend()
    
    plt.subplot(1, 3, 2)
    # BMI distribution
    plt.hist(weight_data['BMI'].dropna(), bins=15, color='green', alpha=0.7, edgecolor='black')
    plt.axvline(18.5, color='blue', linestyle='--', label='Underweight: 18.5')
    plt.axvline(24.9, color='green', linestyle='--', label='Healthy: 24.9')
    plt.axvline(29.9, color='orange', linestyle='--', label='Overweight: 29.9')
    plt.axvline(weight_data['BMI'].mean(), color='red', linestyle='--', 
                label=f'Mean: {weight_data["BMI"].mean():.1f}')
    plt.title('BMI Distribution')
    plt.xlabel('BMI')
    plt.ylabel('Frequency')
    plt.legend()
    
    plt.subplot(1, 3, 3)
    # Weight trends over time for users with multiple entries
    multi_entry_users = weight_data['Id'].value_counts()
    multi_entry_users = multi_entry_users[multi_entry_users > 1].index
    
    for user_id in multi_entry_users[:5]:  # Plot first 5 users with multiple entries
        user_data = weight_data[weight_data['Id'] == user_id].sort_values('Date')
        plt.plot(user_data['Date'], user_data['WeightKg'], marker='o', label=f'User {user_id}')
    
    plt.title('Weight Trends Over Time')
    plt.xlabel('Date')
    plt.ylabel('Weight (kg)')
    plt.xticks(rotation=45)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 10: Correlation Analysis
# Create merged dataset for correlation analysis
# First, aggregate daily steps by user
user_daily_avg = daily_steps.groupby('Id').agg({
    'StepTotal': 'mean',
    'ActivityDay': 'count'
}).rename(columns={'StepTotal': 'AvgDailySteps', 'ActivityDay': 'DaysRecorded'})

# Aggregate sleep data by user
user_sleep_avg = sleep_data.groupby('Id').agg({
    'TotalMinutesAsleep': 'mean',
    'TotalTimeInBed': 'mean',
    'SleepEfficiency': 'mean'
}).rename(columns={
    'TotalMinutesAsleep': 'AvgSleepMinutes',
    'TotalTimeInBed': 'AvgTimeInBed',
    'SleepEfficiency': 'AvgSleepEfficiency'
})

# Merge datasets
correlation_data = user_daily_avg.merge(user_sleep_avg, on='Id', how='inner')

if not weight_data.empty:
    user_weight_avg = weight_data.groupby('Id').agg({
        'WeightKg': 'mean',
        'BMI': 'mean'
    }).rename(columns={'WeightKg': 'AvgWeight', 'BMI': 'AvgBMI'})
    correlation_data = correlation_data.merge(user_weight_avg, on='Id', how='left')

print("Correlation Data Sample:")
print(correlation_data.head())

# Correlation matrix
plt.figure(figsize=(10, 8))
corr_matrix = correlation_data.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
           square=True, linewidths=0.5, fmt='.2f')
plt.title('Correlation Matrix: Activity, Sleep, and Weight Metrics')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Advanced Analytics - Activity Trends Over Time
# Monthly activity trends
monthly_trends = daily_steps.groupby(['Month', 'DayOfWeek']).agg({
    'StepTotal': 'mean'
}).unstack().reindex(['April', 'May'])

plt.figure(figsize=(12, 6))
monthly_trends.plot(kind='bar', figsize=(12, 6))
plt.title('Average Steps by Month and Day of Week')
plt.xlabel('Month')
plt.ylabel('Average Steps')
plt.legend(title='Day of Week', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# User consistency analysis
user_consistency = daily_steps.groupby('Id').agg({
    'StepTotal': ['mean', 'std', 'count']
})
user_consistency.columns = ['Mean_Steps', 'Std_Steps', 'Record_Count']
user_consistency['CV_Steps'] = (user_consistency['Std_Steps'] / user_consistency['Mean_Steps']) * 100

print("User Activity Consistency (Lower CV = More Consistent):")
print(user_consistency.sort_values('CV_Steps').head(10))

In [ ]:
# Cell 12: SQL-like Analysis using Pandas
# Create SQL-like queries using pandas

# 1. Users meeting step recommendations
users_meeting_goal = daily_steps.groupby('Id')['StepTotal'].mean().reset_index()
users_meeting_goal['Meets_Goal'] = users_meeting_goal['StepTotal'] >= 10000

goal_summary = users_meeting_goal['Meets_Goal'].value_counts()
print("Users Meeting 10,000 Steps Goal:")
print(goal_summary)
print(f"Percentage meeting goal: {goal_summary[True]/goal_summary.sum()*100:.1f}%")

# 2. Top performers analysis
top_performers = users_meeting_goal[users_meeting_goal['Meets_Goal'] == True].sort_values('StepTotal', ascending=False)
print("\nTop 5 Performers:")
print(top_performers.head())

# 3. Sleep quality analysis
good_sleepers = sleep_data[sleep_data['TotalMinutesAsleep'] >= 420]  # 7 hours
good_sleep_summary = good_sleepers.groupby('Id').size().reset_index(name='GoodSleepDays')
print(f"\nUsers with good sleep (≥7 hours): {good_sleep_summary.shape[0]}")

In [ ]:
# Cell 13: Strategic Recommendations & Insights
print("="*60)
print("BELLABEAT FITNESS DATA ANALYTICS - KEY INSIGHTS & RECOMMENDATIONS")
print("="*60)

# Key Metrics
avg_steps = daily_steps['StepTotal'].mean()
goal_achievement = (users_meeting_goal['Meets_Goal'].sum() / len(users_meeting_goal)) * 100
avg_sleep = sleep_data['TotalMinutesAsleep'].mean() / 60
sleep_efficiency = sleep_data['SleepEfficiency'].mean()

print(f"\nKEY METRICS:")
print(f"• Average Daily Steps: {avg_steps:.0f}")
print(f"• Users Meeting 10K Steps Goal: {goal_achievement:.1f}%")
print(f"• Average Sleep Duration: {avg_sleep:.1f} hours")
print(f"• Average Sleep Efficiency: {sleep_efficiency:.1f}%")

print(f"\nCRITICAL INSIGHTS:")
print("1. ACTIVITY LEVELS:")
print(f"   - Only {goal_achievement:.1f}% of users meet the recommended 10,000 daily steps")
print(f"   - Average activity ({avg_steps:.0f} steps) is below optimal health levels")

print("\n2. SLEEP PATTERNS:")
print(f"   - Users average {avg_sleep:.1f} hours of sleep (below 7-9 hour recommendation)")
print(f"   - Sleep efficiency of {sleep_efficiency:.1f}% indicates room for improvement")

print("\n3. DATA QUALITY OBSERVATIONS:")
print(f"   - {daily_steps['Id'].nunique()} unique users in steps data")
print(f"   - {sleep_data['Id'].nunique()} unique users in sleep data")
print(f"   - Inconsistent tracking patterns observed across users")

print(f"\nSTRATEGIC RECOMMENDATIONS FOR BELLABEAT:")
print("1. PRODUCT ENHANCEMENTS:")
print("   • Implement smart notifications for sedentary behavior detection")
print("   • Develop personalized step goals based on user history")
print("   • Add sleep quality scoring and improvement suggestions")

print("\n2. MARKETING STRATEGIES:")
print("   • Target users with 'activity challenges' to boost engagement")
print("   • Create educational content on sleep optimization")
print("   • Develop community features for social motivation")

print("\n3. DATA-DRIVEN FEATURES:")
print("   • Predictive analytics for activity slumps")
print("   • Personalized wellness scores")
print("   • Integration with nutritional tracking")

In [ ]:
# Cell 14: Export Analysis Results
# Save cleaned data and analysis results
daily_steps.to_csv('cleaned_daily_steps.csv', index=False)
sleep_data.to_csv('cleaned_sleep_data.csv', index=False)
correlation_data.to_csv('user_correlation_data.csv', index=False)

# Create summary report
summary_report = {
    'Total_Users_Steps': daily_steps['Id'].nunique(),
    'Total_Users_Sleep': sleep_data['Id'].nunique(),
    'Total_Records_Steps': len(daily_steps),
    'Total_Records_Sleep': len(sleep_data),
    'Average_Daily_Steps': avg_steps,
    'Percentage_Meeting_Step_Goal': goal_achievement,
    'Average_Sleep_Hours': avg_sleep,
    'Average_Sleep_Efficiency': sleep_efficiency,
    'Data_Collection_Period_Days': (daily_steps['ActivityDay'].max() - daily_steps['ActivityDay'].min()).days
}

summary_df = pd.DataFrame([summary_report])
summary_df.to_csv('analysis_summary.csv', index=False)

print("Analysis files exported successfully!")
print("Files created:")
print("- cleaned_daily_steps.csv")
print("- cleaned_sleep_data.csv")
print("- user_correlation_data.csv")
print("- analysis_summary.csv")